<a href="https://colab.research.google.com/github/m1Febriansyah/244107020199-ML-12/blob/main/JS04_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import warnings
warnings.filterwarnings('ignore')

# Task 1: K-Means
try:
    df = pd.read_csv('Mall_Customers.csv')
    # Use Annual Income and Spending Score
    X_kmeans = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

    # Find best k using Elbow and Silhouette
    wcss = []
    sil_scores = []
    K = range(2, 11)
    for k in K:
        kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
        kmeans.fit(X_kmeans)
        wcss.append(kmeans.inertia_)
        sil_scores.append(metrics.silhouette_score(X_kmeans, kmeans.labels_))

    best_k = K[np.argmax(sil_scores)]

    # Final model for Task 1
    kmeans_final = KMeans(n_clusters=best_k, init='k-means++', random_state=42)
    y_kmeans = kmeans_final.fit_predict(X_kmeans)

    # Plot K-Means
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(K, wcss, 'bo-')
    plt.title('Elbow Method (Inertia)')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('WCSS')

    plt.subplot(1, 2, 2)
    plt.scatter(X_kmeans[:, 0], X_kmeans[:, 1], c=y_kmeans, cmap='rainbow')
    plt.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1], s=100, c='black', label='Centroids')
    plt.title(f'K-Means Clustering (k={best_k})')
    plt.xlabel('Annual Income (k$)')
    plt.ylabel('Spending Score (1-100)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('kmeans_result.png')
    plt.close()
    print(f"Task 1 completed. Best K based on Silhouette Score: {best_k}")
except Exception as e:
    print(f"Error in Task 1: {e}")

# Task 2: DBSCAN
X, y_true = make_moons(n_samples=1000, noise=0.05, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

# Baseline DBSCAN
db = DBSCAN(eps=0.2, min_samples=5).fit(X_scaled)
core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True
labels = db.labels_

n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

baseline_metrics = {
    'Homogeneity': metrics.homogeneity_score(y_true, labels),
    'Completeness': metrics.completeness_score(y_true, labels),
    'V-measure': metrics.v_measure_score(y_true, labels),
    'ARI': metrics.adjusted_rand_score(y_true, labels),
    'AMI': metrics.adjusted_mutual_info_score(y_true, labels),
    'Silhouette': metrics.silhouette_score(X_scaled, labels) if n_clusters_ > 0 else -1
}

# Plot baseline DBSCAN
plt.figure(figsize=(10, 6))
unique_labels = set(labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1] # Black used for noise
    class_member_mask = (labels == k)

    # Core samples (titik besar)
    xy_core = X_scaled[class_member_mask & core_samples_mask]
    plt.plot(xy_core[:, 0], xy_core[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=10)

    # Non-core samples (titik kecil)
    xy_non_core = X_scaled[class_member_mask & ~core_samples_mask]
    plt.plot(xy_non_core[:, 0], xy_non_core[:, 1], 'o', markerfacecolor=tuple(col), markeredgecolor='k', markersize=4)

plt.title(f'DBSCAN (eps=0.2, min_samples=5)\n{n_clusters_} clusters, {n_noise_} noise points')
plt.savefig('dbscan_baseline.png')
plt.close()

print(f"Task 2 Baseline Metrics:\nClusters: {n_clusters_}\nNoise points: {n_noise_}")
for k, v in baseline_metrics.items():
    print(f"{k}: {v:.4f}")
print("-" * 30)

# DBSCAN Experiments
eps_list = [0.05, 0.1, 0.3, 0.5]
min_samples_list = [3, 10, 20]
results = []

for e in eps_list:
    for m in min_samples_list:
        db_exp = DBSCAN(eps=e, min_samples=m).fit(X_scaled)
        l = db_exp.labels_
        nc = len(set(l)) - (1 if -1 in l else 0)
        nn = list(l).count(-1)

        try:
            sil = metrics.silhouette_score(X_scaled, l) if nc > 0 else -1
        except:
            sil = -1

        hom = metrics.homogeneity_score(y_true, l)
        com = metrics.completeness_score(y_true, l)
        vmeas = metrics.v_measure_score(y_true, l)
        ari = metrics.adjusted_rand_score(y_true, l)
        ami = metrics.adjusted_mutual_info_score(y_true, l)

        results.append({
            'eps': e,
            'min_samples': m,
            'Clusters': nc,
            'Noise': nn,
            'Homogen': round(hom, 4),
            'Complete': round(com, 4),
            'V-Meas': round(vmeas, 4),
            'ARI': round(ari, 4),
            'AMI': round(ami, 4),
            'Silhouette': round(sil, 4)
        })

df_res = pd.DataFrame(results)
print("Task 2 Experiments:")
print(df_res.to_string(index=False))

Task 1 completed. Best K based on Silhouette Score: 5
Task 2 Baseline Metrics:
Clusters: 2
Noise points: 0
Homogeneity: 1.0000
Completeness: 1.0000
V-measure: 1.0000
ARI: 1.0000
AMI: 1.0000
Silhouette: 0.3912
------------------------------
Task 2 Experiments:
 eps  min_samples  Clusters  Noise  Homogen  Complete  V-Meas    ARI    AMI  Silhouette
0.05            3        69    186   0.8156    0.1525  0.2570 0.0300 0.2438      0.1129
0.05           10         3    970   0.0307    0.1268  0.0494 0.0023 0.0459     -0.2942
0.05           20         0   1000   0.0000    1.0000  0.0000 0.0000 0.0000     -1.0000
0.10            3         2     14   0.9862    0.9029  0.9427 0.9722 0.9426      0.2517
0.10           10         7     57   0.9433    0.4095  0.5711 0.5234 0.5698      0.1623
0.10           20         6    850   0.1539    0.1555  0.1547 0.0168 0.1509     -0.3602
0.30            3         2      0   1.0000    1.0000  1.0000 1.0000 1.0000      0.3912
0.30           10         2      0  